In [1]:
import json
import os, cv2, numpy as np
from matplotlib import pyplot as plt

IMG_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\images'
LBL_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\labels'

left, right, center, no_face = [], [], [], []

for f in os.listdir(IMG_DIR):
    if not f.endswith('.jpg'): continue
    
    lbl_path = os.path.join(LBL_DIR, f.rsplit('.', 1)[0] + '.json')
    
    # No face
    if not os.path.exists(lbl_path):
        no_face.append(f)
        continue

    with open(lbl_path) as file:
        label = json.load(file)
    
    pts  = label['shapes'][0]['points']
    x1   = pts[0][0]
    x2   = pts[1][0]
    x_center = (x1 + x2) / 2   # face center x position
    
    # Frame is 640px wide
    if x_center < 213:          # left third
        left.append(f)
    elif x_center > 427:        # right third
        right.append(f)
    else:                       # center third
        center.append(f)

print(f"Left   : {len(left)}")
print(f"Center : {len(center)}")
print(f"Right  : {len(right)}")
print(f"No face: {len(no_face)}")
print(f"Total  : {len(left)+len(center)+len(right)+len(no_face)}")

Left   : 27
Center : 202
Right  : 24
No face: 98
Total  : 351


In [3]:
import os, cv2, json, random

IMG_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\images'
LBL_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\labels'

center = []
for f in os.listdir(IMG_DIR):
    if not f.endswith('.jpg'): continue
    lbl_path = os.path.join(LBL_DIR, f.rsplit('.', 1)[0] + '.json')
    if not os.path.exists(lbl_path): continue
    with open(lbl_path) as file:
        label = json.load(file)
    pts = label['shapes'][0]['points']
    x_center = (pts[0][0] + pts[1][0]) / 2
    if 213 <= x_center <= 427:
        center.append(f)

# Keep 35, delete rest
random.seed(42)
random.shuffle(center)
to_delete = center[35:]

for f in to_delete:
    os.remove(os.path.join(IMG_DIR, f))
    lbl = os.path.join(LBL_DIR, f.rsplit('.', 1)[0] + '.json')
    if os.path.exists(lbl): os.remove(lbl)

print(f"Deleted {len(to_delete)} center images")
print(f"Remaining: {len(os.listdir(IMG_DIR))}")

Deleted 167 center images
Remaining: 184


In [4]:
no_face = [f for f in os.listdir(IMG_DIR) 
           if not os.path.exists(os.path.join(LBL_DIR, f.rsplit('.', 1)[0] + '.json'))]

random.seed(42)
random.shuffle(no_face)
for f in no_face[25:]:
    os.remove(os.path.join(IMG_DIR, f))

print(f"No-face kept: 25")
print(f"Remaining: {len(os.listdir(IMG_DIR))}")

No-face kept: 25
Remaining: 111


In [5]:
import os, cv2, json, random

IMG_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\images'
LBL_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\labels'

random.seed(42)

def trim(file_list, keep=24):
    random.shuffle(file_list)
    for f in file_list[keep:]:
        os.remove(os.path.join(IMG_DIR, f))
        lbl = os.path.join(LBL_DIR, f.rsplit('.', 1)[0] + '.json')
        if os.path.exists(lbl): os.remove(lbl)
    print(f"  kept {min(keep, len(file_list))}, deleted {max(0, len(file_list)-keep)}")

left, center, right, no_face = [], [], [], []

for f in os.listdir(IMG_DIR):
    if not f.endswith('.jpg'): continue
    lbl_path = os.path.join(LBL_DIR, f.rsplit('.', 1)[0] + '.json')
    if not os.path.exists(lbl_path):
        no_face.append(f)
        continue
    with open(lbl_path) as file:
        label = json.load(file)
    pts      = label['shapes'][0]['points']
    x_center = (pts[0][0] + pts[1][0]) / 2
    if x_center < 213:       left.append(f)
    elif x_center > 427:     right.append(f)
    else:                    center.append(f)

print("Trimming:")
print("Left   :", end=" "); trim(left)
print("Center :", end=" "); trim(center)
print("Right  :", end=" "); trim(right)
print("No face:", end=" "); trim(no_face)

print(f"\nFinal: {len(os.listdir(IMG_DIR))} images")

Trimming:
Left   :   kept 24, deleted 3
Center :   kept 24, deleted 11
Right  :   kept 24, deleted 0
No face:   kept 24, deleted 1

Final: 96 images


In [6]:
import os

IMG_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\images'
LBL_DIR = r'C:\\Users\\chsur\\projects\\face\\vgg16-same-size\\up\\data\\labels'

images  = set(f.rsplit('.', 1)[0] for f in os.listdir(IMG_DIR) if f.endswith('.jpg'))
labels  = set(f.rsplit('.', 1)[0] for f in os.listdir(LBL_DIR) if f.endswith('.json'))

orphans = labels - images
for name in orphans:
    os.remove(os.path.join(LBL_DIR, name + '.json'))

print(f"Deleted {len(orphans)} orphan labels")
print(f"Images : {len(os.listdir(IMG_DIR))}")
print(f"Labels : {len(os.listdir(LBL_DIR))}")

Deleted 157 orphan labels
Images : 96
Labels : 72
